# Milo voice render — Chatterbox on a free GPU (Kaggle / Colab)

One notebook, one band per run. Set the three values in the next cell, then **Run All**.
Everything it needs comes from the public repo: the script, the reference voice, the corpus, and
the manifest of clips that already exist (those are skipped). Output is a zip of the NEW clips only —
download it and hand it over to be merged and committed.

| account | CORPUS | BAND |
|---|---|---|
| A | `teen` | `15-16` |
| B | `9-11` then `6-8` | `` |
| C | `teen` | `17-18`, then `12-14` |

Kaggle: Settings → Accelerator → **GPU T4 x2** (or P100), Internet **On**. Session limit is 12 h; `HOURS` stops cleanly before it.
It is resumable: a second run skips whatever the first one produced, once that zip has been merged into the repo.

In [ ]:
CORPUS = '9-11'      # '9-11' | '6-8' | 'teen'
BAND   = ''          # only for teen: '12-14' | '15-16' | '17-18'
VOICE  = 'IvUJKFyjVb5hItY9dJAT'   # Stevie. Teddy (3-5) is XjGYkUkzth8BPs29fmcV — already complete.
HOURS  = 11          # stop cleanly before the session limit
CHUNK  = 100         # lines per process; a zip is refreshed after every chunk

In [ ]:
import os, subprocess, sys, time, pathlib, shutil
WORK = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
os.chdir(WORK)
if not (WORK / 'learn').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RadlorInc/learn.git'], check=True)
REPO = WORK / 'learn'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts', 'setuptools<81', 'imageio-ffmpeg'], check=True)
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
OUT = REPO / 'public/audio' / VOICE          # existing clips live here, so the script skips them
corpus = REPO / 'scripts' / f'.voice-corpus-{CORPUS}.json'
marker = WORK / 'started.marker'; marker.touch()
zip_base = WORK / f'clips-{VOICE[:4]}-{CORPUS}{("-" + BAND) if BAND else ""}'

def new_clips():
    return [p for p in OUT.glob('*.mp3') if p.stat().st_mtime > marker.stat().st_mtime and p.stat().st_size > 1024]

def rezip():
    stage = WORK / 'stage'; shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
    files = new_clips()
    for p in files: shutil.copy2(p, stage / p.name)
    shutil.make_archive(str(zip_base), 'zip', stage)
    return len(files)

t0 = time.time(); chunks = 0
while time.time() - t0 < HOURS * 3600:
    cmd = [sys.executable, str(REPO / 'scripts/chatterbox-render.py'), '--voice', VOICE, '--corpus', str(corpus),
           '--out', str(OUT), '--limit', str(CHUNK)] + (['--band', BAND] if BAND else [])
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    head = next((l for l in r.stdout.splitlines() if 'to render' in l), r.stdout[-300:])
    tail = [l for l in r.stdout.splitlines() if l.strip()[:1].isdigit()][-1:] 
    chunks += 1
    print(f'chunk {chunks}: {head.strip()} | last: {tail[0].strip() if tail else "-"} | zipped {rezip()} new clips | {(time.time()-t0)/60:.0f} min', flush=True)
    if r.returncode != 0: print(r.stderr[-1500:]); time.sleep(20)
    if '· 0 to render' in r.stdout: print('corpus complete'); break
print('done —', rezip(), 'new clips in', str(zip_base) + '.zip')

Download the zip from the Output panel (Kaggle: right sidebar → Output; Colab: Files → download).
Then a second band: change `CORPUS`/`BAND` above and Run All again — the marker resets, so the new zip holds only that band.